# 🛰️ Pista B — El satélite: qué pasó y a quién le pasó
**Código que Salva Vidas · DevFest Santo Domingo 2026 · Taller · Nivel principiante**

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alcastaro/codigo-que-salva-vidas/blob/main/taller/pista_B_satelite.ipynb)

Melissa pasó sobre **Black River**, en el suroeste de Jamaica, el 28 de octubre de 2025. En esta pista usas imágenes de satélite **gratuitas y abiertas** para responder estas preguntas:

| Paso | Pregunta | Herramienta |
|---|---|---|
| 1 | ¿Qué imágenes hay de antes y de después? | Catálogos **STAC** de Sentinel |
| 2 | ¿Qué le pasó a la vegetación? | **Sentinel-2** (óptico) |
| 3 | ¿Dónde se acumuló agua, aunque hubiera nubes? | **Sentinel-1** (radar) |
| 4 | ¿Había gente viviendo ahí? | **AlphaEarth Foundations** (Google DeepMind) |
| 5 | ¿Cómo se lo contamos a la población? | **Gemini** |
| 6 | ¿Y en otro lugar u otro desastre? | Adaptar el cuaderno |

Todo esto **no requiere proyecto de Google Cloud**: los datos son abiertos. Ejecuta cada celda con ▶️ en orden.

## ✅ Antes de empezar (3 minutos)

1. **Inicia sesión** en Colab con tu cuenta de Google (arriba a la derecha). Es lo único que necesitas: no hace falta proyecto de Google Cloud ni tarjeta.
2. **Guarda tu copia:** menú **File → Save a copy in Drive** (*Archivo → Guardar una copia en Drive*). Así tus cambios se guardan y la copia es tuya. Trabaja en esa copia.
3. **Ejecuta las celdas en orden**, de arriba abajo: haz clic en una celda y pulsa ▶️ a su izquierda, o `Shift + Enter`. Cuando la celda termina aparece una ✔️ verde.
4. La primera vez que ejecutes una celda, Colab avisa *"This notebook was not authored by Google"* (*"Este notebook no lo creó Google"*). Pulsa **Run anyway** (*Ejecutar de todos modos*).
5. Las celdas con un formulario a la derecha (desplegables, deslizadores, casillas) se pueden cambiar **sin tocar el código**.

> Los menús aparecen en inglés o en español según el idioma de tu cuenta. Aquí van en inglés, con el español entre paréntesis.

**Todo el código está en este cuaderno.** Cada paso define primero sus funciones (con comentarios que explican qué hace cada parte) y luego las usa. Puedes leerlas, cambiarlas y volver a ejecutarlas.

**¿De dónde salen los datos?** De la carpeta [`data/`](https://github.com/alcastaro/codigo-que-salva-vidas/tree/main/data) del repositorio en GitHub. La función `download()` de la primera celda baja cada archivo la primera vez que hace falta y lo guarda en la carpeta `workshop_data/` de tu sesión (panel 📁 de la izquierda). No tienes que subir ni descargar nada a mano.

In [ ]:
# Paso 0 · Preparar el entorno (~40 s)
#   rasterio           → leer imágenes de satélite (GeoTIFF) directamente desde internet
#   pystac             → entender las respuestas de los catálogos de imágenes (STAC)
#   planetary-computer → firmar los enlaces del radar en Microsoft Planetary Computer
#   google-genai       → hablar con Gemini
!pip -q install -U rasterio pystac planetary-computer google-genai

import json, math, os, time, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

# ---------------------------------------------------------------- descargas
# Todos los datos del taller están en la carpeta data/ del repositorio en GitHub.
# Si haces un fork del repositorio, cambia esta dirección por la de tu copia.
REPO_RAW = os.environ.get("WORKSHOP_RAW", "https://raw.githubusercontent.com/alcastaro/codigo-que-salva-vidas/main")
DATA_DIR = Path("workshop_data")   # aquí se guardan las descargas (panel 📁 de la izquierda)


def download(path, url=None):
    """Baja un archivo una sola vez y devuelve su ruta local.

    path: ruta dentro de data/ del repositorio (por ejemplo "data/melissa/archivo.csv").
    url:  opcional, una dirección completa de otro sitio; se guarda en workshop_data/<path>.
    """
    local = DATA_DIR / path
    if not local.exists():                      # si ya está descargado, no lo baja otra vez
        local.parent.mkdir(parents=True, exist_ok=True)
        request = urllib.request.Request(url or f"{REPO_RAW}/{path}", headers={"User-Agent": "colab-workshop"})
        with urllib.request.urlopen(request, timeout=120) as response:
            local.write_bytes(response.read())
        print(f"⬇️  {path} · {local.stat().st_size / 1e6:.1f} MB")
    return local

# ---------------------------------------------------------------- geometría
def bbox_around(lat, lon, km=10.0):
    """Caja (oeste, sur, este, norte) en grados, de `km` de lado y centrada en el punto.
    Un grado de latitud mide ~111,32 km; uno de longitud mide menos cuanto más lejos del ecuador."""
    dlat = km / 2 / 111.32
    dlon = km / 2 / (111.32 * math.cos(math.radians(lat)))
    return (lon - dlon, lat - dlat, lon + dlon, lat + dlat)


def km2_per_pixel(bbox, shape):
    """Área en km² que representa cada píxel de una imagen de forma `shape` que cubre la caja."""
    lat = (bbox[1] + bbox[3]) / 2
    height_km = (bbox[3] - bbox[1]) * 111.32
    width_km = (bbox[2] - bbox[0]) * 111.32 * math.cos(math.radians(lat))
    return (height_km / shape[0]) * (width_km / shape[1])


def read_window(href, bbox, width=600, nodata=None, resampling="bilinear"):
    """Lee SOLO el recorte de la caja de una imagen remota (Cloud Optimized GeoTIFF), ya en lat/lon.
    No baja la imagen entera (cientos de MB): pide por internet solo los trozos que necesita."""
    import rasterio
    from rasterio.enums import Resampling
    from rasterio.transform import from_bounds
    from rasterio.vrt import WarpedVRT
    # alto en píxeles para que los píxeles salgan cuadrados en el terreno
    height = int(round(width * (bbox[3] - bbox[1]) / (bbox[2] - bbox[0]) / math.cos(math.radians((bbox[1] + bbox[3]) / 2))))
    with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR", GDAL_HTTP_MERGE_CONSECUTIVE_RANGES="YES",
                      GDAL_HTTP_MULTIPLEX="YES", GDAL_HTTP_VERSION="2"):
        with rasterio.open(href) as src:
            options = dict(crs="EPSG:4326", transform=from_bounds(*bbox, width, height),
                           width=width, height=height, resampling=getattr(Resampling, resampling))
            if nodata is not None or src.nodata is not None:
                options["nodata"] = nodata if nodata is not None else src.nodata
            with WarpedVRT(src, **options) as vrt:     # reproyecta "al vuelo" a lat/lon
                return vrt.read()


def resize_mask(mask, shape):
    """Cambia el tamaño de una capa (vecino más cercano) para cruzar capas de distinta resolución."""
    rows = (np.arange(shape[0]) * mask.shape[0] / shape[0]).astype(int)
    cols = (np.arange(shape[1]) * mask.shape[1] / shape[1]).astype(int)
    return mask[np.ix_(rows, cols)]

print("Listo ✅")

### 🔑 Tu API key de Gemini (2 minutos, solo la primera vez)
Si ya la guardaste, salta este bloque. Si no:
1. Abre **[aistudio.google.com/apikey](https://aistudio.google.com/apikey)** con tu cuenta de Google, acepta los términos y pulsa **Create API key**. Es gratis. Copia la key.
2. En Colab, abre el panel de la **llave 🔑 (Secretos)** en la barra de la izquierda.
3. Pulsa **Add new secret** (*Agregar secreto nuevo*). En *Name* escribe exactamente `GOOGLE_API_KEY` y en *Value* pega tu key.
4. Activa el interruptor **Notebook access** (*Acceso al notebook*) de esa fila.
5. Cuando ejecutes la siguiente celda, Colab puede preguntarte si este notebook puede usar el secreto: pulsa **Grant access** (*Otorgar acceso*).

> Atajo: en ese mismo panel, **Gemini API keys → Import key from Google AI Studio** crea y guarda la key por ti.
>
> **Nunca pegues una key dentro de una celda.** Si compartes el notebook o lo subes a GitHub, la key viaja con él y otra persona puede gastar tu cuota.

### 🤖 Conectar con Gemini
La siguiente celda define cómo hablar con Gemini y dos funciones de análisis: `analyze_figure()` le enseña una figura y `analyze_data()` le pasa unos números.
Las usamos después de cada resultado para tener una **segunda lectura**. Gemini puede equivocarse: compárala siempre con la tuya y con el bloque 🔎.

In [ ]:
# ---------------------------------------------------------------- Gemini
import logging
from google import genai
from google.genai import types

# La librería avisa en inglés sobre una función que no usamos; para este taller es solo ruido.
logging.getLogger("google_genai.models").setLevel(logging.ERROR)


def gemini_client():
    """Crea el cliente de la API de Gemini con la key guardada en los Secretos de Colab."""
    try:
        from google.colab import userdata          # solo existe dentro de Colab
        key = userdata.get("GOOGLE_API_KEY")
    except ImportError:                             # fuera de Colab: variable de entorno
        key = os.environ.get("GOOGLE_API_KEY")
    except Exception as error:
        raise RuntimeError("No encuentro tu key. Abre la llave 🔑 (Secretos), crea GOOGLE_API_KEY "
                           "y activa 'Notebook access'.") from error
    if not key:
        raise RuntimeError("Falta GOOGLE_API_KEY. Sácala gratis en aistudio.google.com/apikey.")
    return genai.Client(api_key=key)


# Modelos con cuota gratuita, en orden de preferencia. Se usa el primero que tu key tenga disponible.
PREFERRED_MODELS = {
    "flash": ["gemini-flash-latest", "gemini-flash-lite-latest", "gemini-2.5-flash"],
    "gemma": ["gemma-4-26b-a4b-it", "gemma-4-31b-it"],
}


def pick_model(client, family="flash"):
    """Devuelve el primer modelo preferido que aparece en la lista de modelos de tu key."""
    available = {m.name.split("/")[-1] for m in client.models.list()}
    for name in PREFERRED_MODELS[family]:
        if name in available:
            return name
    return PREFERRED_MODELS[family][0]


def generate(client, model, contents, config=None, attempts=4):
    """Llama al modelo. Si está saturado (503) o llegaste al límite por minuto (429), espera y reintenta.
    En el último intento usa la versión 'lite', que suele tener más capacidad libre."""
    fallback = {"gemini-flash-latest": "gemini-flash-lite-latest"}
    last_error = None
    for attempt in range(attempts):
        current = model if attempt < attempts - 1 else fallback.get(model, model)
        try:
            return client.models.generate_content(model=current, contents=contents, config=config)
        except Exception as error:
            last_error = error
            if not any(code in str(error) for code in ("429", "503", "500", "UNAVAILABLE", "RESOURCE_EXHAUSTED")):
                raise                               # otro tipo de error: lo mostramos tal cual
            print(f"   modelo ocupado, reintento {attempt + 1}…")
            time.sleep(4 * (attempt + 1))
    raise last_error

# ---------------------------------------------------------------- análisis con Gemini
import io
from IPython.display import Markdown, display

ANALYST_RULES = ("Eres analista de datos para gestión de riesgos de desastre. Explica en español sencillo, en 4 a 6 "
                 "viñetas: qué muestra, qué patrón se ve, qué conclusión prudente se puede sacar y qué habría que "
                 "verificar en terreno. Separa lo que se VE de lo que se INTERPRETA. No inventes cifras que no estén "
                 "en la figura o en el contexto. Si algo no se puede saber con estos datos, dilo.")


def analyze_figure(fig, context):
    """Le enseña la figura a Gemini (que entiende imágenes) y le pide que la analice."""
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=80, bbox_inches="tight")   # la figura como imagen PNG
    image = types.Part.from_bytes(data=buffer.getvalue(), mime_type="image/png")
    response = generate(client, model, [image, ANALYST_RULES + "\nContexto: " + context])
    display(Markdown("**🤖 Análisis de Gemini**\n\n" + response.text))


def analyze_data(data, context):
    """Le pasa a Gemini un resultado en números (un diccionario) y le pide que lo analice."""
    text = json.dumps(data, ensure_ascii=False, default=str, indent=1)
    response = generate(client, model, ANALYST_RULES + "\nContexto: " + context + "\nDatos:\n" + text)
    display(Markdown("**🤖 Análisis de Gemini**\n\n" + response.text))

client = gemini_client()
model = pick_model(client, "flash")
print("Gemini listo ✅ · modelo:", model)

In [ ]:
# @title La zona de estudio { run: "auto" }
place = "Black River, Jamaica (huracán Melissa)"  # @param ["Black River, Jamaica (huracán Melissa)", "Mis coordenadas"]
my_lat = 18.4861    # @param {type:"number"}
my_lon = -69.9312   # @param {type:"number"}
my_km = 8           # @param {type:"slider", min:2, max:20, step:1}
# Para analizar otro lugar: elige «Mis coordenadas», escribe latitud y longitud (en Google Maps, clic derecho
# sobre el mapa) y el tamaño de la caja. Luego cambia las fechas en los pasos 1, 2 y 3 por las de tu evento.

EXAMPLE = place.startswith("Black River")      # el ejemplo tiene copias guardadas; otro lugar se lee en vivo
LAT, LON, KM = (18.04, -77.82, 10) if EXAMPLE else (my_lat, my_lon, my_km)
bbox = bbox_around(LAT, LON, KM)
print("Caja (oeste, sur, este, norte):", [round(x, 3) for x in bbox])
if not EXAMPLE:
    print("📍 Lugar propio: los pasos 2, 3 y 4 leerán los satélites en vivo (puede tardar unos minutos).")

## Paso 1 · Buscar imágenes como un profesional: STAC

**STAC** (*SpatioTemporal Asset Catalog*) es el estándar abierto para buscar imágenes de satélite: le das un lugar y unas fechas, y te devuelve las escenas disponibles.
Una búsqueda STAC es una simple petición web (`POST /search`) con un JSON. Usamos dos catálogos públicos:
- **Earth Search** (Element 84, sobre AWS) para **Sentinel-2**: fotos ópticas de 10 m, como una cámara, con las nubes como problema.
- **Microsoft Planetary Computer** para **Sentinel-1**: radar, que **atraviesa las nubes** y funciona de noche.

Los dos satélites son del programa europeo **Copernicus**, y sus datos son libres.

In [ ]:
EARTH_SEARCH = "https://earth-search.aws.element84.com/v1"
PLANETARY = "https://planetarycomputer.microsoft.com/api/stac/v1"


def search_stac(collection, bbox, dates, catalog=EARTH_SEARCH, max_cloud=None):
    """Busca escenas en un catálogo STAC. dates = "AAAA-MM-DD/AAAA-MM-DD". Devuelve una tabla."""
    import pystac
    start, end = dates.split("/")
    body = {"collections": [collection], "bbox": list(bbox), "limit": 100,
            "datetime": f"{start}T00:00:00Z/{end}T23:59:59Z"}
    if max_cloud is not None:                       # filtro opcional: solo escenas con pocas nubes
        body["query"] = {"eo:cloud_cover": {"lt": max_cloud}}
    response = requests.post(f"{catalog}/search", json=body, timeout=60)
    response.raise_for_status()
    items = [pystac.Item.from_dict(f) for f in response.json()["features"]]
    rows = [{"date": it.datetime.strftime("%Y-%m-%d %H:%M"),
             "cloud_%": round(it.properties.get("eo:cloud_cover", float("nan")), 1),
             "orbit": it.properties.get("sat:relative_orbit"),     # solo en radar: la "pasada" del satélite
             "id": it.id, "item": it} for it in items]
    return pd.DataFrame(rows).sort_values("date").reset_index(drop=True) if rows else pd.DataFrame()


s2 = search_stac("sentinel-2-l2a", bbox, "2025-10-15/2025-11-10", max_cloud=60)
print("Sentinel-2 (óptico):"); display(s2.drop(columns="item"))
s1 = search_stac("sentinel-1-rtc", bbox, "2025-10-10/2025-11-05", catalog=PLANETARY)
print("Sentinel-1 (radar):"); display(s1.drop(columns="item"))

#### 📖 Cómo leer las tablas
- Cada fila es una **escena**: una foto que tomó el satélite al pasar. `date` está en hora UTC.
- `cloud_%` es el porcentaje de la escena cubierto por nubes. Por encima de 30–40 % suele ser inservible para ver el suelo.
- La misma fecha puede aparecer **dos veces** en Sentinel-2: el lugar cae en el borde de dos cuadrículas (*tiles*) y las unimos.
- `orbit` (radar) identifica la pasada del satélite. Para comparar antes y después hay que usar **la misma órbita**, porque así el satélite mira con el mismo ángulo.

#### 🔎 Qué vemos en el ejemplo
- Sentinel-2 pasó varias veces entre el 15 y el 18 de octubre con 16–40 % de nubes; cada fecha aparece varias veces porque Black River cae entre cuadrículas.
- En el radar hay dos órbitas: la **113** (hacia las 11:00 UTC) y la **150** (hacia las 23:18 UTC). La 150 pasó el **17** y el **29 de octubre**: es el par de antes y después, con el mismo ángulo de vista.

**🧠 Para pensar:** quita `max_cloud=60` y vuelve a ejecutar. Mira el 25 y el 28 de octubre, los días del huracán: **100 % de nubes**. Por eso necesitamos el radar.

## Paso 2 · Antes y después en color real

Leemos cuatro **bandas** (colores) de Sentinel-2: rojo, verde, azul e infrarrojo cercano. Con las tres primeras armamos la foto en color.
El **NDVI** (*índice de vegetación de diferencia normalizada*) compara el rojo con el infrarrojo: las plantas sanas reflejan mucho infrarrojo y poco rojo.
`NDVI = (infrarrojo − rojo) / (infrarrojo + rojo)`: cerca de 1 es vegetación sana; cerca de 0, suelo, agua o daño.

In [ ]:
def mosaic(items, band, bbox, width=600):
    """Une varias escenas del mismo día en una imagen: rellena los huecos de una con la otra."""
    out = None
    for it in items:
        a = read_window(it.assets[band].href, bbox, width, nodata=0)[0].astype("float32")
        a[a == 0] = np.nan                          # 0 = fuera de la escena
        out = a if out is None else np.where(np.isnan(out), a, out)
    return out


def best_per_tile(items):
    """Si una cuadrícula tiene dos escenas el mismo día, se queda con la de menos nubes."""
    best = {}
    for it in items:
        tile = it.properties.get("grid:code") or it.id.split("_")[1]
        if tile not in best or it.properties["eo:cloud_cover"] < best[tile].properties["eo:cloud_cover"]:
            best[tile] = it
    return list(best.values())


def read_sentinel2(items, bbox, width=600):
    """Color real (valores 0–1) y NDVI de un día de Sentinel-2."""
    items = best_per_tile(items)
    b = {name: mosaic(items, name, bbox, width) for name in ("red", "green", "blue", "nir")}
    rgb = np.clip(np.dstack([b["red"], b["green"], b["blue"]]) / 3000, 0, 1)   # 3000 = brillo de referencia
    ndvi = (b["nir"] - b["red"]) / (b["nir"] + b["red"] + 1e-6)
    return rgb, ndvi


def load_saved_copy(path="data/sentinel/black_river.npz"):
    """Lo mismo que leen las celdas en vivo para Black River, guardado de antemano en el repositorio.
    Se guardó comprimido en enteros; aquí se vuelve a convertir a las unidades originales."""
    c = np.load(download(path))
    to_float = lambda x, k=1: np.where(x == -128, np.nan, x.astype("float32") / k)
    return {"rgb_before": c["rgb_before"] / 255, "rgb_after": c["rgb_after"] / 255,
            "ndvi_before": to_float(c["ndvi_before"], 100), "ndvi_after": to_float(c["ndvi_after"], 100),
            "db_before": to_float(c["db_before"]), "db_after": to_float(c["db_after"])}

In [ ]:
# @title Antes y después { run: "auto" }
LIVE = False                # @param {type:"boolean"}
BEFORE_DATE = "2025-10-20"  # @param {type:"string"}
AFTER_DATE = "2025-10-30"   # @param {type:"string"}
# LIVE = True lee las imágenes ahora mismo desde AWS (30–60 s) usando las fechas de arriba.
# LIVE = False usa una copia idéntica de Black River guardada en GitHub (ignora las fechas).
def scenes_on(table, day):
    """Las escenas de un día; si no hay, un error que dice qué hacer."""
    chosen = list(table[table.date.str.startswith(day)].item)
    if not chosen:
        raise ValueError(f"No hay escena el {day} en tu zona. Elige una fecha de la tabla del paso 1.")
    return chosen

if LIVE or not EXAMPLE:   # otro lugar: siempre en vivo, la copia guardada solo es de Black River
    rgb_before, ndvi_before = read_sentinel2(scenes_on(s2, BEFORE_DATE), bbox)
    rgb_after, ndvi_after = read_sentinel2(scenes_on(s2, AFTER_DATE), bbox)
else:
    saved = load_saved_copy()
    rgb_before, rgb_after = saved["rgb_before"], saved["rgb_after"]
    ndvi_before, ndvi_after = saved["ndvi_before"], saved["ndvi_after"]
ndvi_loss = ndvi_before - ndvi_after

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(rgb_before); ax[0].set_title(f"{BEFORE_DATE} (antes)")
ax[1].imshow(rgb_after); ax[1].set_title(f"{AFTER_DATE} (después)")
im = ax[2].imshow(ndvi_loss, cmap="RdYlGn_r", vmin=-0.4, vmax=0.4); ax[2].set_title("Pérdida de vegetación (NDVI antes − después)")
plt.colorbar(im, ax=ax[2], shrink=0.7)
for a in ax: a.axis("off")
plt.show()

#### 📖 Cómo leer las imágenes
- Las dos primeras son **fotos en color real**: el norte está arriba y la imagen cubre la caja de 10 km.
- En el mapa de pérdida, **rojo** = la vegetación está menos verde que antes (hojas arrancadas, árboles caídos, barro); **verde** = más verde que antes; **amarillo** = sin cambio.
- Las **nubes** y sus sombras también salen como "pérdida" o "ganancia": antes de sacar conclusiones, mira si en esa zona había nubes en alguna de las dos fotos.

#### 🔎 Qué vemos en el ejemplo
- La foto del 30 de octubre es más parda y apagada que la del 20: el viento arrancó hojas y ramas.
- El NDVI medio de la zona baja de **0,56 a 0,22**, y unos **8 de cada 10 píxeles** pierden más de 0,2: una pérdida de vegetación generalizada.
- Parte de esa pérdida puede ser bruma o nubes finas en la foto de después. Por eso se confirma con el radar del paso 3.

In [ ]:
analyze_figure(fig, f"Sentinel-2 en color real antes ({BEFORE_DATE}) y después ({AFTER_DATE}) de un huracán, y mapa de "
                    f"pérdida de vegetación (NDVI antes − después; rojo = pérdida). Caja de {KM} km en {LAT}, {LON}.")

## Paso 3 · El radar ve el agua a través de las nubes

El radar de Sentinel-1 lanza microondas y mide cuánto rebota (*retrodispersión*), en **decibelios (dB)**. El **agua quieta** actúa como un espejo y devuelve muy poco, así que sale **oscura** (valores muy negativos).
Regla simple: si un píxel pasa de "claro" antes a "oscuro" después, ahí apareció **agua nueva**.

In [ ]:
def read_radar_db(item, bbox, width=600):
    """Retrodispersión VV de Sentinel-1 en decibelios. El agua quieta sale muy oscura."""
    import planetary_computer
    from scipy.ndimage import median_filter
    href = planetary_computer.sign(item.assets["vv"].href)   # Planetary Computer pide firmar el enlace
    a = read_window(href, bbox, width)[0].astype("float32")
    a[a <= 0] = np.nan
    return median_filter(10 * np.log10(a), size=3)   # a dB, y quitamos el "granulado" típico del radar

In [ ]:
# @title El radar antes y después { run: "auto" }
LIVE = False                        # @param {type:"boolean"}
ORBIT = 150                         # @param {type:"integer"}
RADAR_BEFORE = "2025-10-17"         # @param {type:"string"}
RADAR_AFTER = "2025-10-29"          # @param {type:"string"}
WATER_DB = -18                      # @param {type:"slider", min:-24, max:-12, step:1}
# LIVE = True lee el radar desde Planetary Computer (de 20 s a varios minutos, según la red).
if LIVE or not EXAMPLE:
    same_orbit = s1[s1.orbit == ORBIT]
    db_before = read_radar_db(scenes_on(same_orbit, RADAR_BEFORE)[0], bbox)
    db_after = read_radar_db(scenes_on(same_orbit, RADAR_AFTER)[0], bbox)
else:
    saved = load_saved_copy()
    db_before, db_after = saved["db_before"], saved["db_after"]

new_water = (db_after < WATER_DB) & ~(db_before < WATER_DB)   # oscuro después, pero no antes
water_km2 = new_water.sum() * km2_per_pixel(bbox, new_water.shape)

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(db_before, cmap="gray", vmin=-25, vmax=0); ax[0].set_title(f"Radar {RADAR_BEFORE}")
ax[1].imshow(db_after, cmap="gray", vmin=-25, vmax=0); ax[1].set_title(f"Radar {RADAR_AFTER}")
ax[2].imshow(db_after, cmap="gray", vmin=-25, vmax=0)
ax[2].imshow(np.ma.masked_where(~new_water, new_water), cmap="cool", alpha=0.9)
ax[2].set_title(f"Agua nueva ≈ {water_km2:.1f} km²")
for a in ax: a.axis("off")
plt.show()

#### 📖 Cómo leer el radar
- En gris: **oscuro** = superficie lisa que refleja la señal lejos del satélite (agua quieta, asfalto); **claro** = superficie rugosa o edificios.
- En el tercer panel, lo **rosado/celeste** es "agua nueva": oscuro después y no antes. El título da el área total en km².
- Mueve `WATER_DB`: un umbral más bajo (−22) es más estricto y marca menos agua; uno más alto (−14) marca más, pero con más falsos positivos.

**🧠 Para pensar:** ¿ves "agua nueva" en el **mar**? No es una inundación. Después del huracán había viento, el mar se picó y eso cambia el rebote del radar.
Un umbral simple tiene límites. Los sistemas reales, como Flood Hub, usan modelos entrenados y máscaras de agua permanente.

#### 🔎 Qué vemos en el ejemplo
- Con el umbral de −18 dB aparecen unos **2,8 km² de agua nueva**.
- Mira dónde se concentra: lo esperable tras un huracán es junto al río y en las zonas bajas del humedal, donde se acumula el agua.
- Las manchas sobre el mar no son inundación: es el viento sobre el oleaje.

In [ ]:
analyze_figure(fig, f"Radar Sentinel-1 (VV, dB) antes ({RADAR_BEFORE}) y después ({RADAR_AFTER}) de un huracán. Tercer "
                    f"panel: en color, 'agua nueva' = más oscuro que {WATER_DB} dB después pero no antes. "
                    f"Área estimada: {water_km2:.1f} km².")

## Paso 4 · ¿Qué había en el suelo? AlphaEarth Foundations

**AlphaEarth Foundations** (Google DeepMind) resume un año de observaciones de satélite (óptico, radar, relieve, clima…) en **64 números por cada píxel de 10 m**. Esos 64 números son un **embedding**, una especie de "huella digital" del terreno.
Dos lugares parecidos (dos barrios, dos manglares) tienen huellas parecidas. Así se puede **buscar por similitud** sin entrenar ningún modelo.

Los embeddings anuales son abiertos en **Source Cooperative** (CC BY 4.0). Para Black River usamos una **copia** de 2024 guardada en el repositorio, porque leer el original tarda unos minutos.

In [ ]:
# ---------------------------------------------------------------- AlphaEarth Foundations
# Los embeddings anuales están publicados como GeoTIFF en Source Cooperative (CC BY 4.0).
# Este índice (una tabla pequeña guardada en el repo) dice qué archivo cubre cada lugar y cada año.
AEF_INDEX = pd.read_parquet(download("data/aef_index_light.parquet"))


def dequantize(q):
    """Los 64 números se guardan como enteros de 8 bits (-127…127) para ocupar menos.
    Se recuperan con (v / 127,5)² × signo(v). El valor -128 significa «sin dato»."""
    x = q.astype("float32")
    x = np.sign(x) * (x / 127.5) ** 2
    x[q == -128] = np.nan
    return x


def alphaearth_live(bbox, year, width=300):
    """Lee en vivo los embeddings de un año para la caja: un arreglo (64, alto, ancho).
    Tarda alrededor de un minuto por año en una caja de 3–4 km."""
    lon, lat = (bbox[0] + bbox[2]) / 2, (bbox[1] + bbox[3]) / 2
    row = AEF_INDEX[(AEF_INDEX.year == year) & (AEF_INDEX.wgs84_west <= lon) & (AEF_INDEX.wgs84_east >= lon)
                    & (AEF_INDEX.wgs84_south <= lat) & (AEF_INDEX.wgs84_north >= lat)]
    if row.empty:
        raise ValueError(f"No hay embeddings de AlphaEarth de {year} para ese lugar (hay de 2017 a 2025).")
    raw = read_window("/vsicurl/" + row.path.iloc[0], bbox, width, nodata=-128, resampling="nearest")
    return dequantize(raw)


def embedding_rgb(*embs):
    """Convierte 64 números por píxel en un color: resume los 64 en las 3 direcciones que más varían
    (análisis de componentes principales, PCA). Colores parecidos = terrenos parecidos.
    Los colores en sí no significan nada (no es 'verde = bosque').

    Si le pasas VARIAS imágenes (por ejemplo 2017 y 2025), calcula UNA sola conversión para todas:
    así el mismo tipo de terreno sale del mismo color en todos los años y se pueden comparar a simple vista.
    Con una imagen devuelve una imagen; con varias, una lista."""
    c = embs[0].shape[0]
    flat = [np.nan_to_num(e.reshape(c, -1).T) for e in embs]       # píxeles × 64, una tabla por imagen
    stack = np.concatenate(flat)
    mean = stack.mean(0)
    _, _, vt = np.linalg.svd((stack - mean)[:: max(1, len(stack) // 30000)], full_matrices=False)
    proj = [(x - mean) @ vt[:3].T for x in flat]                    # las 3 direcciones principales = R, G, B
    lo, hi = np.percentile(np.concatenate(proj), [2, 98], axis=0)   # mismo rango de brillo para todas
    out = [np.clip((pr - lo) / (hi - lo), 0, 1).reshape(e.shape[1], e.shape[2], 3) for pr, e in zip(proj, embs)]
    return out[0] if len(out) == 1 else out


def change_between(a, b):
    """1 − similitud coseno entre las huellas de dos años, píxel a píxel.
    0 = el terreno no cambió; cuanto más alto, más cambió."""
    dot = np.nansum(a * b, 0)
    norms = np.sqrt(np.nansum(a ** 2, 0)) * np.sqrt(np.nansum(b ** 2, 0))
    return 1 - dot / np.where(norms == 0, np.nan, norms)


def load_embeddings(path):
    """Embeddings guardados en el repositorio (mismo formato comprimido que el original)."""
    return dequantize(np.load(download(path))["q"])


def similarity(emb, vector):
    """Similitud coseno de cada píxel con un vector de referencia: 1 = idéntico, 0 = nada que ver."""
    norms = np.sqrt(np.nansum(emb ** 2, axis=0)) * np.linalg.norm(vector)
    return np.einsum("chw,c->hw", np.nan_to_num(emb), vector) / np.where(norms == 0, np.nan, norms)


def vector_at(emb, bbox, lat, lon, radius_px=2):
    """La huella promedio alrededor de un punto: «esto es lo que busco»."""
    _, h, w = emb.shape
    row = int((bbox[3] - lat) / (bbox[3] - bbox[1]) * h)       # la fila 0 está al norte
    col = int((lon - bbox[0]) / (bbox[2] - bbox[0]) * w)
    patch = emb[:, max(row - radius_px, 0):row + radius_px + 1, max(col - radius_px, 0):col + radius_px + 1]
    return np.nanmean(patch.reshape(64, -1), axis=1)


def near(mask, bbox, meters):
    """Marca todo lo que queda a menos de `meters` de la máscara (la «engorda» en círculo)."""
    from scipy.ndimage import binary_dilation
    pixel_m = math.sqrt(km2_per_pixel(bbox, mask.shape)) * 1000
    steps = max(1, int(round(meters / pixel_m)))
    yy, xx = np.ogrid[-steps:steps + 1, -steps:steps + 1]
    return binary_dilation(mask, structure=(xx ** 2 + yy ** 2) <= steps ** 2)


if EXAMPLE:
    emb = load_embeddings("data/alphaearth/black_river_2024.npz")   # copia guardada (año anterior al huracán)
else:
    emb = alphaearth_live(bbox, 2024, width=300)                     # tu lugar, en vivo (unos minutos)
print("Forma:", emb.shape, "→ 64 números por píxel")

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(rgb_before); ax[0].set_title("Sentinel-2 (lo que ve una cámara)")
ax[1].imshow(embedding_rgb(emb)); ax[1].set_title("AlphaEarth 2024 (64 números resumidos en 3 colores)")
for a in ax: a.axis("off")
plt.show()

#### 📖 Cómo leer la imagen de AlphaEarth
Los colores **no** significan "verde = bosque": solo indican que dos zonas del **mismo color tienen huellas parecidas**. Fíjate en que el humedal, el pueblo y el mar quedan separados con bordes nítidos, aunque en la foto se confundan.

Ahora le damos a AlphaEarth un ejemplo de **zona construida** (el centro de Black River) y le pedimos que encuentre todo lo que se parece.
Después **cruzamos** esa capa con el agua nueva del radar: **zonas construidas a menos de 250 m de agua nueva**. Así se mide la *exposición* en una evaluación rápida.

In [ ]:
# @title Busca lugares parecidos a este punto { run: "auto" }
# El punto de referencia: un ejemplo del tipo de terreno que buscas (aquí, el centro del pueblo).
# Con «Mis coordenadas» se usa el centro de tu caja; cámbialo por un barrio, un cultivo o un manglar.
REF_LAT, REF_LON = (18.033, -77.858) if EXAMPLE else (LAT, LON)
SIMILARITY_MIN = 0.90  # @param {type:"slider", min:0.6, max:0.98, step:0.01}
DISTANCE_M = 250       # @param {type:"slider", min:50, max:1000, step:50}

reference = vector_at(emb, bbox, REF_LAT, REF_LON)
sim = similarity(emb, reference)
built = sim > SIMILARITY_MIN                                   # "se parece al pueblo"

water_on_emb = resize_mask(new_water, built.shape)            # radar en la misma cuadrícula que AlphaEarth
exposed = built & near(water_on_emb, bbox, meters=DISTANCE_M)
px_km2 = km2_per_pixel(bbox, built.shape)

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(sim, cmap="viridis", vmin=0.5, vmax=1); ax[0].set_title("Parecido al pueblo (1 = idéntico)")
ax[1].imshow(rgb_after); ax[1].set_title(f"Construido a <{DISTANCE_M} m de agua nueva ≈ {exposed.sum() * px_km2:.2f} km²")
layer = resize_mask(exposed, rgb_after.shape[:2])
ax[1].imshow(np.ma.masked_where(~layer, layer), cmap="autumn", alpha=0.9)
for a in ax: a.axis("off")
plt.show()
print(f"Zona parecida a pueblo: {built.sum() * px_km2:.2f} km² · de ella, junto a agua nueva: {exposed.sum() * px_km2:.2f} km²")

#### 📖 Cómo leer el resultado
- Izquierda: **amarillo** = muy parecido al centro del pueblo; **morado** = nada parecido.
- Derecha: en rojo, las zonas **parecidas a pueblo** que tienen **agua nueva** a menos de la distancia elegida. Es una estimación de exposición, no un conteo de casas dañadas.
- Si subes `SIMILARITY_MIN`, solo quedan las zonas casi idénticas al punto de referencia (menos área, más segura); si la bajas, entran también zonas rurales o caminos.

#### 🔎 Qué vemos en el ejemplo
- Unos **3,95 km²** de la caja se parecen al centro del pueblo (similitud mayor de 0,90).
- De ellos, **0,18 km²** (alrededor del 5 %) tienen agua nueva a menos de 250 m: son las manzanas que una brigada revisaría primero.
- Es una estimación rápida: no cuenta casas ni personas, y el radar no ve el agua entre edificios ni bajo los árboles.

In [ ]:
analyze_figure(fig, f"Izquierda: similitud (embeddings de AlphaEarth) de cada píxel con el centro de un pueblo. Derecha: en "
                    f"rojo, zonas parecidas a pueblo (similitud > {SIMILARITY_MIN}) a menos de {DISTANCE_M} m de agua "
                    f"nueva detectada por radar tras un huracán. Parecido a pueblo: {built.sum() * px_km2:.2f} km²; "
                    f"expuesto: {exposed.sum() * px_km2:.2f} km².")

## Paso 5 · De los mapas a una alerta
Le pasamos los números a **Gemini** para que redacte un aviso claro: qué pasó, dónde y qué hacer. La celda siguiente define la conexión y las reglas de redacción.

In [ ]:
# ---------------------------------------------------------------- la alerta
# Las "instrucciones de sistema" son las reglas fijas del modelo. Cámbialas y vuelve a ejecutar.
ALERT_INSTRUCTIONS = """Eres el redactor de alertas de una oficina de protección civil del Caribe.
Recibes datos técnicos y escribes UNA alerta para la población, en español sencillo.
Reglas:
- Se tiene que entender en 3 segundos: primero QUÉ pasa, luego DÓNDE, luego QUÉ HACER.
- Máximo 280 caracteres en el campo 'mensaje'. Sin tecnicismos (nada de nudos, hPa, NDVI ni dB).
- No inventes datos que no estén en la entrada. Si la certeza es baja, dilo.
- Incluye los campos del Protocolo Común de Alertas (CAP): severidad, urgencia y certeza.
"""

# El "esquema" obliga al modelo a responder con un JSON con estos campos exactos.
# severidad, urgencia y certeza usan los valores oficiales del estándar CAP (en inglés).
ALERT_SCHEMA = {
    "type": "object",
    "properties": {
        "titulo": {"type": "string"},
        "mensaje": {"type": "string"},
        "que_hacer": {"type": "array", "items": {"type": "string"}},
        "severidad": {"type": "string", "enum": ["Extreme", "Severe", "Moderate", "Minor", "Unknown"]},
        "urgencia": {"type": "string", "enum": ["Immediate", "Expected", "Future", "Past", "Unknown"]},
        "certeza": {"type": "string", "enum": ["Observed", "Likely", "Possible", "Unlikely", "Unknown"]},
    },
    "required": ["titulo", "mensaje", "que_hacer", "severidad", "urgencia", "certeza"],
}


def write_alert(client, model, technical_data):
    """Convierte un diccionario de números en una alerta estructurada (un dict con los campos del esquema)."""
    content = "Datos técnicos:\n" + json.dumps(technical_data, ensure_ascii=False, default=str, indent=1)
    if "gemma" in model:
        # Gemma no admite instrucciones de sistema ni esquema por la API: lo pedimos todo en el texto.
        prompt = (ALERT_INSTRUCTIONS + "\nResponde SOLO con un JSON con las claves "
                  + ", ".join(ALERT_SCHEMA["required"]) + ". 'que_hacer' es una lista de frases cortas. "
                  + "severidad: Extreme|Severe|Moderate|Minor|Unknown; urgencia: Immediate|Expected|Future|Past|Unknown; "
                  + "certeza: Observed|Likely|Possible|Unlikely|Unknown (en inglés, tal cual).\n\n" + content)
        text = generate(client, model, prompt).text.strip()
        return json.loads(text.removeprefix("```json").removeprefix("```").removesuffix("```"))
    config = types.GenerateContentConfig(system_instruction=ALERT_INSTRUCTIONS,
                                         response_mime_type="application/json",
                                         response_json_schema=ALERT_SCHEMA,
                                         temperature=0.2)   # temperatura baja = respuestas más estables
    return json.loads(generate(client, model, content, config).text)


def show_alert(alert):
    """Dibuja la alerta como una notificación de teléfono."""
    from IPython.display import HTML, display
    colors = {"Extreme": "#ea4335", "Severe": "#f9ab00", "Moderate": "#4285f4", "Minor": "#34a853"}
    color = colors.get(alert.get("severidad"), "#1e1e1e")
    steps = alert.get("que_hacer", [])
    if isinstance(steps, str):          # a veces Gemma devuelve una frase en vez de una lista
        steps = [s.strip() for s in steps.replace(";", ".").split(".") if s.strip()]
    items = "".join(f"<li>{s}</li>" for s in steps)
    display(HTML(f"""
    <div style="max-width:380px;border:2px solid #1e1e1e;border-radius:28px;padding:20px 24px;
                font-family:'Google Sans',system-ui,sans-serif;background:#fff;color:#1e1e1e">
      <div style="display:inline-block;background:{color};color:#fff;border-radius:999px;padding:4px 14px;
                  font-size:13px">{alert.get('severidad')} · {alert.get('urgencia')} · {alert.get('certeza')}</div>
      <h3 style="margin:12px 0 6px">{alert.get('titulo')}</h3>
      <p style="font-size:16px;margin:0 0 8px">{alert.get('mensaje')}</p>
      <ul style="margin:0;padding-left:20px">{items}</ul>
    </div>"""))

print("Funciones de Gemini listas ✅")

In [ ]:
client = gemini_client()
model = pick_model(client, "flash")
technical_data = {
    "evento": "Huracán Melissa, inundación posterior",
    "zona": "Black River y alrededores, parroquia de St. Elizabeth, Jamaica",
    "fuente": f"Satélites Sentinel-1 (radar, {RADAR_AFTER}) y Sentinel-2, análisis automático",
    "agua_nueva_km2": round(float(water_km2), 1),
    f"zona_construida_a_menos_de_{DISTANCE_M}m_de_agua_nueva_km2": round(float(exposed.sum() * px_km2), 2),
    "vegetacion": "pérdida fuerte de vegetación tras el huracán",
    "advertencia": "Estimación automática por satélite, sin validar en terreno",
}
show_alert(write_alert(client, model, technical_data))

#### 📖 Cómo leer la alerta
- La **etiqueta de color** resume los tres campos del estándar CAP: **severidad** (*Extreme* = amenaza extraordinaria para la vida; *Severe* = amenaza significativa; *Moderate*, *Minor*), **urgencia** (*Immediate* = actuar ya; *Expected* = en la próxima hora o pocas horas; *Future* = más adelante) y **certeza** (*Observed* = ya está pasando; *Likely* = más del 50 %; *Possible* = menos del 50 %).
- Revisa que el **mensaje** diga primero qué pasa, luego dónde y luego qué hacer, y que no tenga cifras técnicas.
- Comprueba que **no inventó nada**: todo lo que diga tiene que estar en el diccionario de datos que le pasaste. Si ves un dato nuevo (una hora, un refugio, un número), es una alucinación.
- Cada vez que ejecutes la celda la redacción cambia un poco: es normal en un modelo de lenguaje.

## Paso 6 · 🔧 Adapta el cuaderno a otro lugar u otro desastre

El cuaderno sirve para cualquier lugar del mundo y cualquier evento desde 2017. Estos son los cambios, en orden:

1. **La zona** (celda *La zona de estudio*): pon `LAT`, `LON` del centro del lugar (en Google Maps, clic derecho sobre el mapa y copia las coordenadas) y el tamaño `KM`. Empieza con 5–10 km: más grande tarda más.
2. **Las fechas de búsqueda** (celda del paso 1): cambia `"2025-10-15/2025-11-10"` y `"2025-10-10/2025-11-05"` por un rango que cubra unos días **antes y después** de tu evento. Ejecuta y mira las tablas.
3. **Antes y después en óptico** (paso 2): activa `LIVE` y escribe en `BEFORE_DATE` y `AFTER_DATE` dos fechas de la tabla de Sentinel-2 con **pocas nubes**.
4. **Radar** (paso 3): activa `LIVE`, elige en la tabla de Sentinel-1 una `ORBIT` que aparezca **antes y después** del evento, y escribe esas dos fechas.
5. **AlphaEarth** (paso 4): la copia guardada solo cubre Black River. Para tu lugar, usa la celda de abajo para leer el año anterior al evento en vivo (unos minutos para 10 km; usa `width=300` para ir más rápido), y cambia `REF_LAT`, `REF_LON` por un punto del tipo de terreno que buscas (un barrio, un cultivo, un manglar).
6. **La alerta** (paso 5): cambia `"evento"`, `"zona"` y las fechas en `technical_data`.

Ideas de eventos para probar: inundaciones del **río Yaque del Norte** (noviembre de 2023), **tormenta Franklin** en RD (agosto de 2023), **huracán Fiona** en el este de RD (septiembre de 2022).

In [ ]:
# @title AlphaEarth en vivo para tu zona
# Lee AlphaEarth en vivo para la zona actual (bbox) y la usa en el paso 4. Tarda varios minutos con 10 km:
# por eso está apagado. Actívalo solo cuando hayas cambiado la zona.
# Después, vuelve a ejecutar la celda «Busca lugares parecidos» con tu punto de referencia.
LIVE = False      # @param {type:"boolean"}
EMB_YEAR = 2024   # @param {type:"integer"}
if LIVE:
    emb = alphaearth_live(bbox, EMB_YEAR, width=300)
    plt.imshow(embedding_rgb(emb)); plt.title(f"AlphaEarth {EMB_YEAR}"); plt.axis("off"); plt.show()
else:
    print("Activa LIVE para leer AlphaEarth en vivo en tu zona.")

## 🏁 Reto · ¿Qué cambió en ocho años en tu barrio?
Compara la huella de AlphaEarth de 2017 con la de 2025 en un lugar de **3 km** (cada año tarda alrededor de un minuto).
Prueba la orilla del Ozama o la expansión de Santiago.

In [ ]:
# @title Compara dos años en tu lugar
my_lat, my_lon = 18.4861, -69.9312  # @param {type:"raw"}
year_a, year_b = 2017, 2025         # @param {type:"raw"}
my_bbox = bbox_around(my_lat, my_lon, km=3)

emb_a = alphaearth_live(my_bbox, year_a, width=300)
emb_b = alphaearth_live(my_bbox, year_b, width=300)
change = change_between(emb_a, emb_b)
rgb_a, rgb_b = embedding_rgb(emb_a, emb_b)   # la MISMA escala de color para los dos años

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(rgb_a); ax[0].set_title(str(year_a))
ax[1].imshow(rgb_b); ax[1].set_title(str(year_b))
im = ax[2].imshow(change, cmap="magma", vmin=0, vmax=0.6); ax[2].set_title("Cuánto cambió (claro = mucho)")
plt.colorbar(im, ax=ax[2], shrink=0.7)
for a in ax: a.axis("off")
plt.show()

**📖 Cómo leerlo:** en el mapa de cambio, **negro** = la huella del terreno es igual en los dos años; **amarillo claro** = cambió mucho (construcción nueva, deforestación, cambio de cultivo, un cauce que se movió). Las dos primeras usan la **misma escala de colores**: el mismo color es el mismo tipo de terreno en los dos años, así que puedes compararlas a simple vista.

In [ ]:
analyze_figure(fig, f"AlphaEarth {year_a} y {year_b} con la misma escala de color (colores iguales = terreno parecido) y mapa "
                    f"de cambio (1 − similitud; claro = cambió mucho). Caja de 3 km en {my_lat}, {my_lon}.")

## 🆘 Si algo falla

| Lo que ves | Qué hacer |
|---|---|
| `NameError: name '...' is not defined` | Te saltaste una celda: cada paso usa funciones definidas antes. Usa **Runtime → Run before** (*Entorno de ejecución → Ejecutar anteriores*). |
| `ModuleNotFoundError` | Ejecuta otra vez la primera celda (la que instala librerías). |
| `No encuentro tu key` o `Falta GOOGLE_API_KEY` | Revisa el bloque 🔑: el nombre debe ser exactamente `GOOGLE_API_KEY` y *Notebook access* debe estar activo. |
| `429` o `RESOURCE_EXHAUSTED` | Llegaste al límite gratuito por minuto. Espera un minuto y vuelve a ejecutar la celda. |
| `503` o `UNAVAILABLE` | El modelo está saturado. El código reintenta solo; si aun así falla, prueba de nuevo en un minuto. |
| `HTTP Error 404` al descargar | La dirección no existe: revisa la fecha o el identificador que escribiste. |
| Una celda tarda mucho | Las celdas con `LIVE = True` leen datos en directo y dependen de la red. Ponlas en `False` para usar la copia guardada en GitHub. |
| Todo se rompió | **Runtime → Restart session** y ejecuta de nuevo desde la primera celda. |

---
**Fuentes y licencias.** Sentinel-1 y Sentinel-2: Copernicus (ESA/UE), datos libres; Sentinel-2 L2A vía Earth Search (Element 84) y Sentinel-1 RTC vía Microsoft Planetary Computer.
AlphaEarth Foundations Satellite Embedding: Google DeepMind / Google Earth Engine, copia en Source Cooperative (CC BY 4.0).
Los análisis de este notebook son **didácticos**: un umbral de radar no sustituye una evaluación de daños oficial.